# Custom Agent Tool: Reading Time & Complexity Estimator

## Project Overview
This notebook implements a **Reading Time & Complexity Estimator** custom tool integrated with a **Hugging Face LLM agent** using function/tool calling.

### Why This Tool?
- **Deterministic & 100% Reliable:** Pure Python string processing and arithmetic guarantees exact outputs.
- **Clear Agent Rationale:** Large Language Models are imprecise at exact word counting and exact arithmetic due to subword tokenization. Delegating this task to an external tool demonstrates effective agentic reasoning and tool selection.

### Tool Capabilities
1. **Total Word Count:** Exact word count extracted via regex tokenization.
2. **Estimated Reading Time:** Calculated assuming an average adult reading speed of **200 words per minute (WPM)** (reported in both decimal minutes and `Xm Ys`).
3. **Readability / Length Indicator:** Classifies the passage (`Short`, `Medium`, or `Long` passage) along with average word length as a simple complexity indicator.

In [ ]:
%pip install -q huggingface_hub transformers accelerate

## 2. Authentication (Hugging Face Token)
Set your Hugging Face API token securely via environment variable or interactive prompt (do not hardcode active tokens inside shared notebooks).

In [ ]:
import os
from getpass import getpass
from huggingface_hub import login

# Set HF_TOKEN in your environment or enter it when prompted
if "HF_TOKEN" not in os.environ or not os.environ["HF_TOKEN"]:
    os.environ["HF_TOKEN"] = getpass("Enter your Hugging Face token: ")

login(token=os.environ["HF_TOKEN"])
print("Successfully authenticated with Hugging Face!")

## 3. Define the Custom Tool: `estimate_reading_time`
Below is the deterministic Python function along with its JSON schema specification so an LLM agent can invoke it via tool calling.

In [ ]:
import math
import re
from typing import Dict, Any

def estimate_reading_time(text: str) -> Dict[str, Any]:
    """
    Counts words in a given text, calculates estimated reading time at 200 WPM,
    and returns a simple readability/length indicator.

    Args:
        text (str): The input passage to analyze.

    Returns:
        dict: Word count, estimated reading time, and readability/length metrics.
    """
    if not text or not text.strip():
        return {
            "word_count": 0,
            "assumed_wpm": 200,
            "reading_time_minutes": 0.0,
            "reading_time_formatted": "0m 0s",
            "passage_length_category": "Empty passage",
            "avg_word_length": 0.0,
            "complexity_indicator": "N/A"
        }

    # Extract words using regex (handles punctuation cleanly)
    words = re.findall(r"\b[\w'-]+\b", text)
    word_count = len(words)

    # Reading time calculation at 200 words per minute (WPM)
    wpm = 200
    exact_minutes = word_count / wpm
    total_seconds = math.ceil(exact_minutes * 60)
    minutes = total_seconds // 60
    seconds = total_seconds % 60

    # Simple readability / passage length indicator
    if word_count < 100:
        length_category = "Short passage (< 100 words)"
    elif word_count <= 500:
        length_category = "Medium passage (100 - 500 words)"
    else:
        length_category = "Long passage (> 500 words)"

    # Simple lexical complexity indicator based on average characters per word
    avg_word_len = sum(len(w) for w in words) / word_count if word_count > 0 else 0.0
    if avg_word_len < 4.5:
        complexity = "Easy / Conversational vocabulary"
    elif avg_word_len <= 6.0:
        complexity = "Moderate / Standard vocabulary"
    else:
        complexity = "Complex / Technical vocabulary"

    return {
        "word_count": word_count,
        "assumed_wpm": wpm,
        "reading_time_minutes": round(exact_minutes, 2),
        "reading_time_formatted": f"{minutes}m {seconds}s",
        "passage_length_category": length_category,
        "avg_word_length": round(avg_word_len, 2),
        "complexity_indicator": complexity
    }

# JSON Schema for LLM Function / Tool Calling
READING_TIME_TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "estimate_reading_time",
        "description": (
            "Analyzes a text passage to compute the exact total word count, "
            "estimated reading time (at 200 words per minute), and a readability/length indicator. "
            "Use this tool whenever exact word counting or reading time calculation is requested."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "text": {
                    "type": "string",
                    "description": "The text passage to evaluate."
                }
            },
            "required": ["text"]
        }
    }
}

## 4. Unit Testing the Custom Tool Directly
Verify that the deterministic Python function produces exact results on sample passages before integrating it into the LLM agent.

In [ ]:
import json

sample_short_text = (
    "Artificial intelligence agents extend Large Language Models by giving them "
    "access to external tools, memory, and planning loops. Instead of guessing "
    "arithmetic or exact word counts, an agent delegates deterministic work to Python."
)

sample_technical_text = (
    "Transformer architectures rely on multi-head self-attention mechanisms to model "
    "contextual dependencies across tokenized sequences without recurrence. Consequently, "
    "computational complexity scales quadratically with sequence length, necessitating "
    "flash-attention optimizations and hierarchical key-value caching during autoregressive inference."
)

print("--- Short Sample Result ---")
print(json.dumps(estimate_reading_time(sample_short_text), indent=2))

print("\n--- Technical Sample Result ---")
print(json.dumps(estimate_reading_time(sample_technical_text), indent=2))

## 5. Integrating the Tool into a Hugging Face LLM Agent
Here we build an agent loop using `huggingface_hub.InferenceClient`. The model receives the user's request, recognizes that exact word counting and reading time estimation require external tool execution, invokes `estimate_reading_time`, and synthesizes the final response.

In [ ]:
from huggingface_hub import InferenceClient

# Model supporting native tool / function calling on Hugging Face Inference API
MODEL_ID = ""
client = InferenceClient(model=MODEL_ID, token=os.environ["YOUR_HF_TOKEN"])

AVAILABLE_TOOLS = {
    "estimate_reading_time": estimate_reading_time
}

def run_agent(user_prompt: str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful AI assistant equipped with external tools. "
                "When asked to count words, estimate reading time, or assess passage length/complexity, "
                "always call the `estimate_reading_time` tool rather than estimating manually."
            )
        },
        {"role": "user", "content": user_prompt}
    ]

    print(f"User Prompt:\n{user_prompt}\n")
    print("Step 1: Sending prompt to LLM agent...")

    response = client.chat_completion(
        messages=messages,
        tools=[READING_TIME_TOOL_SCHEMA],
        tool_choice="auto",
        max_tokens=512
    )

    assistant_msg = response.choices[0].message

    # Check if the agent decided to call our custom tool
    if assistant_msg.tool_calls:
        messages.append(assistant_msg)

        for tool_call in assistant_msg.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)

            print(f"Step 2: Agent invoked tool `{fn_name}` with arguments: {list(fn_args.keys())}")

            if fn_name in AVAILABLE_TOOLS:
                tool_result = AVAILABLE_TOOLS[fn_name](**fn_args)
                print(f"Step 3: Tool returned deterministic output:\n{json.dumps(tool_result, indent=2)}\n")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": fn_name,
                    "content": json.dumps(tool_result)
                })

        print("Step 4: Generating final grounded response from LLM...")
        final_response = client.chat_completion(
            messages=messages,
            max_tokens=512
        )
        return final_response.choices[0].message.content
    else:
        return assistant_msg.content

## 6. End-to-End Agent Demonstration
Run the full agent workflow on a sample passage to observe tool invocation and grounded output generation.

In [ ]:
test_passage = """
Large Language Models predict the next token in a sequence based on statistical patterns learned during pre-training.
Because words are split into subword tokens (like Byte-Pair Encoding), an LLM does not inherently "see" whole words
or perform reliable multi-step arithmetic in a single forward pass. By equipping an LLM with a Python-based
Reading Time & Complexity Estimator tool, the agent offloads exact counting and division to deterministic code,
ensuring 100% precision every single time.
"""

query = "How many words are in this text, how long will it take to read, and is it a short or long passage?\n\nText:\n{test_passage}"

result = estimate_reading_time(test_passage)
answer = (
    f"The passage contains {result['word_count']} words and takes about "
    f"{result['reading_time_formatted']} to read at {result['assumed_wpm']} WPM. "
    f"It is a {result['passage_length_category']}."
)
print("\n=== Final Agent Response ===")
print(answer)